# 00. 데이터 로딩과 무결성 확인

원본 CSV 4개를 병합하고 dtype을 최적화해 parquet으로 저장한다.
**CSV 재파싱은 매번 1분 이상 걸리므로 여기서 한 번만 수행**하고,
이후 모든 노트북은 parquet을 읽는다.

산출물: `data/processed/train.parquet`, `test.parquet`

In [ ]:
import sys
from pathlib import Path

# src/ 를 import 경로에 추가 (uv pip install -e . 를 했다면 불필요)
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 110

from frauddetectionreport import config, data
config.ensure_dirs()

## 원본 파일 확인

`data/raw/` 에 아래 4개 파일이 있어야 한다.
없다면 [Kaggle 대회 페이지](https://www.kaggle.com/competitions/ieee-fraud-detection)에서
받아 배치한다 (대회 규정 동의 필요).

In [ ]:
for f in sorted(config.RAW_DIR.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")

## 병합과 dtype 최적화

`transaction` + `identity` 를 `TransactionID` 로 left join 한다.
identity는 일부 거래에만 존재하므로 결측이 대량 발생하는데,
이는 결손이 아니라 **'기기 정보가 수집되지 않은 거래'라는 정보 자체**다.

dtype 최적화(float64→float32 등)로 메모리를 절반 이하로 줄인다.
434개 변수를 반복 실험할 것이므로 이 차이가 이후 작업 속도를 좌우한다.

In [ ]:
%%time
train = data.build_parquet("train", overwrite=False)
test = data.build_parquet("test", overwrite=False)

print(f"train {train.shape}   test {test.shape}")
print(f"메모리: train {train.memory_usage(deep=True).sum()/1e9:.2f} GB")

## 기본 무결성

In [ ]:
from frauddetectionreport.config import TARGET, ID_COL, TIME_COL, AMOUNT_COL

checks = {
    "train 행 수": len(train),
    "test 행 수": len(test),
    "TransactionID 중복 (train)": train[ID_COL].duplicated().sum(),
    "TransactionID 중복 (test)": test[ID_COL].duplicated().sum(),
    "train/test ID 교집합": len(set(train[ID_COL]) & set(test[ID_COL])),
    "사기 건수": int(train[TARGET].sum()),
    "사기율": round(train[TARGET].mean(), 5),
    "금액 결측": int(train[AMOUNT_COL].isna().sum()),
    "금액 최소": round(train[AMOUNT_COL].min(), 2),
    "금액 최대": round(train[AMOUNT_COL].max(), 2),
}
for k, v in checks.items():
    print(f"{k:28s} {v}")

### 시간 범위

`TransactionDT` 는 기준 시점으로부터의 초 단위 경과값이다.
**train과 test가 시간적으로 겹치지 않아야 한다** — 겹친다면 대회 설정에 대한
이해가 틀린 것이므로 검증 설계를 재고해야 한다.

In [ ]:
for name, df in [("train", train), ("test", test)]:
    t = df[TIME_COL]
    print(f"{name:6s}  {t.min():>12,.0f} ~ {t.max():>12,.0f}   "
          f"({(t.max()-t.min())/86400:6.1f}일)")

gap = (test[TIME_COL].min() - train[TIME_COL].max()) / 86400
print(f"\ntrain 종료 → test 시작 간격: {gap:.1f}일")
assert test[TIME_COL].min() > train[TIME_COL].max(), "시간 겹침 — 검증 설계 재고 필요"

## 변수군 구성

IEEE-CIS는 변수 대부분이 익명화되어 있고 접두사로만 성격을 짐작할 수 있다.

In [ ]:
groups = data.column_groups(train)
for g, cols in groups.items():
    print(f"{g:8s} {len(cols):4d}개   {cols[:6]}{' ...' if len(cols) > 6 else ''}")

## 결측 구조

V 변수군은 결측이 **블록 단위**로 발생한다 (같이 비고 같이 채워짐).
이 구조 자체가 '어느 수집 경로를 탔는가'라는 정보이므로, 단순 대치가 아니라
블록 패턴을 피처로 쓰는 것을 04에서 검토한다.

In [ ]:
miss = data.missing_summary(train)
print(f"결측 0%   : {(miss.missing_rate == 0).sum():3d}개")
print(f"결측 90%+ : {(miss.missing_rate > 0.9).sum():3d}개")
display(miss.head(15))

## 요약

- parquet 캐시 생성 완료 → 이후 노트북은 `data.load("train")` 으로 즉시 로딩
- train/test 시간 분리 확인 → **시간 기반 split 사용의 근거** (03에서 상세)
- 다음: `01_eda_overview` 에서 구조 파악